### ИМПОРТЫ

In [1173]:
from pathlib import Path

import pandas as pd
import numpy as np

ROOT = Path.cwd()
if not (ROOT / "train_data").exists() and (ROOT.parent / "train_data").exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / "train_data" / "raw_data"
train = pd.read_csv(DATA_DIR / "train.csv")


In [1174]:
transactions = pd.read_csv(DATA_DIR / "transactions.csv")
bureau = pd.read_csv(DATA_DIR / "bureau.csv")
previous_loans = pd.read_csv(DATA_DIR / "previous_loans.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

### EDA (Изучение структуры данных, формирование сводной таблицы со всеми потенциальными признаками для анализа во 2 части) | 1 Часть

#### 1. Изучение структуры данных

In [1175]:
pd.set_option('display.max_columns', None)
train.head()

,application_id,client_id,employment_type,hash_id,region_coefficient_extended,age,incoming_amount,internal_decision_code,monthly_income,application_date,education,requested_product,channel,loan_amount,marketing_segment,post_loan_collection_score,region_coefficient,region,interest_rate,months_at_job,dependents,days_until_first_overdue,loan_term_months,siberia_northern_score,target
0,107941,7941,employee,61441390.0,1.0,37.0,30401.28,NaN,30401.28,2025-08-11,bachelor,refinance,partner,63464.49,segment_011,800.87,1.0,east,0.3062,31.0,0.0,61.0,24.0,0.336102,1
1,101163,1163,employee,67321556.0,0.0,60.0,26024.64,manual_review_bad,26024.64,2025-12-29,school,card,mobile,113513.59,segment_067,721.57,1.0,ural,0.1383,90.0,0.0,82.0,6.0,0.560893,1
2,100583,583,contractor,26719968.0,0.0,37.0,21389.83,vip,21389.83,2025-11-09,college,cash,NaN,48993.42,NaN,NaN,1.0,west,0.1729,57.0,1.0,999.0,9.0,NaN,0
3,104082,4082,NaN,45184358.0,0.0,NaN,31617.26,approved_auto,NaN,2025-02-23,college,card,web,77765.78,segment_089,170.91,1.0,ural,0.3585,19.0,0.0,999.0,36.0,NaN,0
4,108413,8413,employee,58267524.0,0.0,60.0,18000.00,approved_auto,18000.00,2025-12-07,bachelor,card,mobile,97254.10,segment_063,NaN,1.0,west,0.1472,405.0,0.0,730.0,9.0,0.741053,0


In [1176]:
train.describe()

,application_id,client_id,hash_id,region_coefficient_extended,age,incoming_amount,monthly_income,loan_amount,post_loan_collection_score,region_coefficient,interest_rate,months_at_job,dependents,days_until_first_overdue,loan_term_months,siberia_northern_score,target
count,6488.000000,6488.000000,6.071000e+03,6027.000000,6039.000000,6055.000000,6047.000000,5932.000000,4950.000000,6132.0,6113.000000,6102.000000,6018.000000,4632.000000,5961.000000,6056.000000,6488.000000
mean,104496.386868,4496.386868,5.503946e+07,0.005973,43.636529,42386.646533,42300.330501,77673.946999,490.591644,1.0,0.270003,102.636676,0.805085,580.034111,20.849019,0.505414,0.342170
std,2592.661534,2592.661534,2.594531e+07,0.077061,13.600516,21867.000385,21759.337138,45118.826338,225.343839,0.0,0.096328,86.869559,0.900978,436.833759,11.907587,0.288577,0.474473
min,100001.000000,1.000000,1.000185e+07,0.000000,21.000000,18000.000000,18000.000000,20000.000000,0.000000,1.0,0.080000,0.000000,0.000000,1.000000,6.000000,0.000109,0.000000
25%,102256.500000,2256.500000,3.237164e+07,0.000000,32.000000,26815.515000,26777.040000,47056.442500,320.677500,1.0,0.193000,36.000000,0.000000,97.000000,12.000000,0.257933,0.000000
50%,104518.500000,4518.500000,5.501823e+07,0.000000,44.000000,37773.110000,37717.510000,67344.210000,472.205000,1.0,0.250800,80.000000,1.000000,730.000000,18.000000,0.509269,0.000000
75%,106714.500000,6714.500000,7.818919e+07,0.000000,55.000000,52218.765000,52181.895000,96352.650000,659.425000,1.0,0.332300,149.000000,1.000000,1000.000000,24.000000,0.754349,1.000000
max,109000.000000,9000.000000,9.999550e+07,1.000000,67.000000,282686.980000,282686.980000,496150.840000,1000.000000,1.0,0.480000,506.000000,5.000000,1002.000000,48.000000,0.999899,1.000000


In [1177]:
test.head()

,application_id,client_id,region_coefficient_extended,loan_term_months,siberia_northern_score,hash_id,loan_amount,post_loan_collection_score,months_at_job,channel,education,internal_decision_code,monthly_income,region,employment_type,days_until_first_overdue,incoming_amount,age,dependents,interest_rate,marketing_segment,region_coefficient,requested_product,application_date
0,102531,2531,0.0,18.0,NaN,NaN,73071.51,NaN,38.0,call_center,bachelor,NaN,34183.79,east,employee,NaN,34183.79,NaN,0.0,0.2146,segment_112,1.0,refinance,2025-04-25
1,107213,7213,0.0,12.0,0.317813,30012008.0,65408.73,NaN,23.0,partner,school,NaN,18645.00,north,unemployed,NaN,18645.00,64.0,0.0,0.2851,segment_023,1.0,refinance,2025-05-07
2,100238,238,0.0,36.0,0.902039,92419982.0,NaN,NaN,12.0,office,bachelor,NaN,52931.84,ural,employee,NaN,52931.84,21.0,1.0,0.3810,segment_020,1.0,card,2025-02-26
3,104918,4918,0.0,6.0,0.737848,76137566.0,NaN,NaN,179.0,web,college,NaN,NaN,center,employee,NaN,37301.16,55.0,0.0,0.1471,segment_057,1.0,auto,2025-12-07
4,106480,6480,0.0,12.0,0.746599,21845266.0,76253.20,NaN,153.0,mobile,bachelor,NaN,71928.29,center,NaN,NaN,71928.29,55.0,1.0,0.1840,segment_138,1.0,cash,2025-05-09


In [1178]:
test.describe()

,application_id,client_id,region_coefficient_extended,loan_term_months,siberia_northern_score,hash_id,loan_amount,post_loan_collection_score,months_at_job,internal_decision_code,monthly_income,days_until_first_overdue,incoming_amount,age,dependents,interest_rate,region_coefficient
count,2520.000000,2520.000000,2314.000000,2360.000000,2352.000000,2.291000e+03,2351.000000,0.0,2329.000000,0.0,2284.000000,0.0,2318.000000,2322.000000,2365.000000,2317.000000,2351.0
mean,104509.963095,4509.963095,0.003889,20.505508,0.503897,5.490572e+07,78265.656159,NaN,102.082439,NaN,42128.436581,NaN,42065.310738,43.946598,0.801691,0.267957,1.0
std,2612.549464,2612.549464,0.062257,11.599870,0.290296,2.614188e+07,45163.700324,NaN,87.850046,NaN,21195.124881,NaN,21131.202253,13.502092,0.903554,0.094078,0.0
min,100002.000000,2.000000,0.000000,6.000000,0.000896,1.004359e+07,20000.000000,NaN,0.000000,NaN,18000.000000,NaN,18000.000000,21.000000,0.000000,0.094700,1.0
25%,102218.250000,2218.250000,0.000000,12.000000,0.251946,3.197252e+07,46888.620000,NaN,35.000000,NaN,26683.157500,NaN,26614.375000,32.000000,0.000000,0.192700,1.0
50%,104454.500000,4454.500000,0.000000,18.000000,0.510454,5.578085e+07,67730.790000,NaN,79.000000,NaN,37825.845000,NaN,37680.640000,44.000000,1.000000,0.247400,1.0
75%,106875.000000,6875.000000,0.000000,24.000000,0.755153,7.774957e+07,98272.030000,NaN,147.000000,NaN,51860.830000,NaN,51848.940000,55.000000,1.000000,0.332400,1.0
max,108998.000000,8998.000000,1.000000,48.000000,0.997266,9.995308e+07,537558.790000,NaN,524.000000,NaN,196085.590000,NaN,196085.590000,67.000000,5.000000,0.480000,1.0


In [1179]:
set(train.columns) - set(test.columns)

{'target'}

In [1180]:
# Признаки, кандидаты на исключение:
exclude_train_features = ['application_id', 'hash_id']

In [1181]:
transactions.head()

,client_id,application_id,transaction_date,transaction_category,amount,merchant_risk_level
0,1,100001,2025-10-22,transfer,-1538.95,5.0
1,1,100001,2025-07-10,cash_withdrawal,-1460.46,1.0
2,1,100001,2025-11-01,utilities,-988.50,2.0
3,1,100001,2025-10-10,salary,8578.27,3.0
4,1,100001,2025-12-24,entertainment,-2015.69,2.0


In [1182]:
transactions.describe()

,client_id,application_id,amount,merchant_risk_level
count,353300.000000,353300.000000,341886.000000,338285.000000
mean,4498.879856,104498.879856,-351.372813,2.281629
std,2597.796811,2597.796811,11658.936876,1.219794
min,1.000000,100001.000000,-99514.770000,1.000000
25%,2249.000000,102249.000000,-3573.080000,1.000000
50%,4495.000000,104495.000000,-1840.155000,2.000000
75%,6748.000000,106748.000000,-850.822500,3.000000
max,9000.000000,109000.000000,788077.580000,5.000000


In [1183]:
bureau.head()

,client_id,bureau_account_id,account_type,opened_days_ago,credit_limit,current_balance,max_dpd_last_12m,bureau_status
0,1,B1_0,credit_card,2410.0,20474.70,5760.10,0.0,active
1,2,B2_1,NaN,2983.0,61748.02,47175.10,1.0,active
2,3,B3_2,auto,943.0,93069.06,40725.25,15.0,active
3,3,B3_3,mortgage,2355.0,173852.51,57974.36,7.0,closed
4,3,B3_4,credit_card,302.0,24037.79,7801.91,1.0,closed


In [1184]:
bureau.describe()

,client_id,opened_days_ago,credit_limit,current_balance,max_dpd_last_12m
count,24689.000000,24037.000000,23862.000000,24209.000000,23832.000000
mean,4503.892584,1530.544952,73635.370562,31605.814437,9.955102
std,2601.993418,851.999018,52976.082696,29566.123851,20.058778
min,1.000000,60.000000,5162.010000,208.920000,0.000000
25%,2246.000000,788.000000,38598.330000,12849.940000,0.000000
50%,4537.000000,1526.000000,59774.735000,23324.850000,1.000000
75%,6766.000000,2274.000000,92573.527500,40686.930000,7.000000
max,9000.000000,2999.000000,823503.730000,589438.390000,90.000000


In [1185]:
previous_loans.head()

,client_id,previous_loan_id,previous_amount,previous_term_months,closed_days_ago,was_overdue,max_overdue_days
0,2,L2_0,51911.71,3.0,986.0,1.0,3.0
1,2,L2_1,64582.12,24.0,1172.0,0.0,0.0
2,3,L3_2,27102.97,3.0,744.0,0.0,0.0
3,4,L4_3,56399.68,12.0,139.0,1.0,15.0
4,4,L4_4,73117.38,18.0,548.0,0.0,0.0


In [1186]:
previous_loans.describe()

,client_id,previous_amount,previous_term_months,closed_days_ago,was_overdue,max_overdue_days
count,12762.000000,12445.000000,12117.000000,12246.000000,12496.000000,12318.000000
mean,4512.050306,56621.215050,15.324090,813.172709,0.215589,3.866131
std,2598.207487,33839.413872,10.647548,457.405745,0.411247,12.819714
min,2.000000,6101.880000,3.000000,20.000000,0.000000,0.000000
25%,2232.000000,33506.610000,6.000000,417.000000,0.000000,0.000000
50%,4545.500000,48665.570000,12.000000,813.500000,0.000000,0.000000
75%,6739.750000,70704.430000,24.000000,1215.000000,0.000000,0.000000
max,9000.000000,527095.240000,36.000000,1599.000000,1.000000,90.000000


Проверим ключи на уникальность, чтобы понимать, как связывать данные. 

In [1187]:
train['client_id'].nunique(), train['application_id'].nunique(), len(train)

(6480, 6480, 6488)

Обнаружили дубликаты, проверим их.

In [1188]:
# train[train.duplicated(keep=False)].sort_values('application_id')

In [1189]:
df = train.drop_duplicates()
df['client_id'].nunique(), df['application_id'].nunique(), len(df)

(6480, 6480, 6480)

Проверим уникальность пар 'client_id' и 'application_id' в df.

In [1190]:
df[["application_id", "client_id"]].drop_duplicates().shape

(6480, 2)

Проверим ключи в transactions.

In [1191]:
print("Строк:", len(transactions))
print("Уникальных application_id:", transactions["application_id"].nunique())
print("Уникальных client_id:", transactions["client_id"].nunique())

Строк: 353300
Уникальных application_id: 9000
Уникальных client_id: 9000


Проверим количество транзакций на 1 заявку.

In [1192]:
transactions.groupby("application_id").size().describe()

count    9000.000000
mean       39.255556
std         6.609336
min        19.000000
25%        35.000000
50%        39.000000
75%        44.000000
max        76.000000
dtype: float64

Создадим полный список заявок и проверим, насколько сопоставлены ключи.

In [1193]:
all_applications = pd.concat(
    [
        train[["application_id", "client_id"]],
        test[["application_id", "client_id"]],
    ],
    ignore_index=True,
)

In [1194]:
unknown_applications = set(transactions["application_id"]) - set(
    all_applications["application_id"]
)

len(unknown_applications)

0

In [1195]:
# transactions[transactions.duplicated(keep=False)]

df_transactions = transactions.drop_duplicates()
len(df_transactions["application_id"].unique())

9000

Дальше проверяем bureau и previous_loans

Проверяем наличие пропусков:

In [1196]:
bureau["client_id"].isna().sum(), previous_loans["client_id"].isna().sum()

(np.int64(0), np.int64(0))

Проверяем, сколько записей приходится на клиента:

In [1197]:
bureau.groupby("client_id").size().describe()

count    8542.000000
mean        2.890307
std         1.483793
min         1.000000
25%         2.000000
50%         3.000000
75%         4.000000
max        10.000000
dtype: float64

In [1198]:
previous_loans.groupby("client_id").size().describe()

count    6885.000000
mean        1.853595
std         1.005247
min         1.000000
25%         1.000000
50%         2.000000
75%         2.000000
max         8.000000
dtype: float64

Проверяем клиентов, которых нет в train/test:

In [1199]:
bureau_unknown_clients = set(bureau["client_id"]) - set(
    all_applications["client_id"]
)

previous_unknown_clients = set(previous_loans["client_id"]) - set(
    all_applications["client_id"]
)

print(len(bureau_unknown_clients))
print(len(previous_unknown_clients))

0
0


Проверем наликилие client_id в 3-х дополнительных таблицах (проверка покрытия)

In [1200]:
print(
    "Есть transactions:",
    df["client_id"].isin(transactions["client_id"]).mean()
)

print(
    "Есть bureau:",
    df["client_id"].isin(bureau["client_id"]).mean()
)

print(
    "Есть previous_loans:",
    df["client_id"].isin(previous_loans["client_id"]).mean()
)

Есть transactions: 1.0
Есть bureau: 0.9479938271604939
Есть previous_loans: 0.7675925925925926


Это значит, что для всех client_id есть транзакции. Для 95% есть внешняя кредитная история. И для 77% есть предыдущие займы.

Теперь необходимо понять, как агрегировать данные из дополнительных таблиц. При этом сразу нужно заметить потенциальны leakege данные в таблице транзакций. Если транзакция совершена после applicattion_date, то это сигнал из будущего. 

Отфильтруем все нерелевантные транзакции.

In [1201]:
df.head(2)

,application_id,client_id,employment_type,hash_id,region_coefficient_extended,age,incoming_amount,internal_decision_code,monthly_income,application_date,education,requested_product,channel,loan_amount,marketing_segment,post_loan_collection_score,region_coefficient,region,interest_rate,months_at_job,dependents,days_until_first_overdue,loan_term_months,siberia_northern_score,target
0,107941,7941,employee,61441390.0,1.0,37.0,30401.28,NaN,30401.28,2025-08-11,bachelor,refinance,partner,63464.49,segment_011,800.87,1.0,east,0.3062,31.0,0.0,61.0,24.0,0.336102,1
1,101163,1163,employee,67321556.0,0.0,60.0,26024.64,manual_review_bad,26024.64,2025-12-29,school,card,mobile,113513.59,segment_067,721.57,1.0,ural,0.1383,90.0,0.0,82.0,6.0,0.560893,1


In [1202]:
df_transactions.head(2)

,client_id,application_id,transaction_date,transaction_category,amount,merchant_risk_level
0,1,100001,2025-10-22,transfer,-1538.95,5.0
1,1,100001,2025-07-10,cash_withdrawal,-1460.46,1.0


In [1203]:
df_for_trans_merge = df[['client_id', 'application_date']]

df_transactions_old = df_transactions.copy()

df_transactions = df_transactions_old.merge(df_for_trans_merge, on='client_id', how='inner')

In [1204]:
df_transactions.columns

Index(['client_id', 'application_id', 'transaction_date',
       'transaction_category', 'amount', 'merchant_risk_level',
       'application_date'],
      dtype='str')

In [1205]:
df_transactions['transaction_date'] = pd.to_datetime(df_transactions['transaction_date'])
df_transactions['application_date'] = pd.to_datetime(df_transactions['application_date'])

In [1206]:
df_transactions = df_transactions[df_transactions['transaction_date'] < df_transactions['application_date']]

In [1207]:
df_transactions[df_transactions['client_id'] == 4].head().sort_values(['client_id', 'transaction_date', 'application_date'], ascending=[True, False, False])

,client_id,application_id,transaction_date,transaction_category,amount,merchant_risk_level,application_date
38,4,100004,2025-06-03,transfer,-1769.63,5.0,2025-07-03
34,4,100004,2025-05-29,salary,25444.58,NaN,2025-07-03
37,4,100004,2025-03-30,entertainment,-3600.51,3.0,2025-07-03
35,4,100004,2025-02-20,transfer,-4201.50,3.0,2025-07-03
36,4,100004,2025-02-03,cash_withdrawal,-8251.70,3.0,2025-07-03


Итак, мы изменили файл df_transaction тем, что отсекли будущие операции. Дальше перейдем к собиранию признков.

#### 2. Собираем принаки из df_transactions

Сперва проанализируем таблицу df_transactions. Здесь больше всего интересует показатель merchant_risk_level, transaction_category, amount.

In [1208]:
df_transactions.describe()

,client_id,application_id,transaction_date,amount,merchant_risk_level,application_date
count,229106.000000,229106.000000,229106,221744.000000,219409.000000,229106
mean,4490.505521,104490.505521,2025-04-01 12:25:44.298272,-345.391387,2.283981,2025-07-01 00:57:06.494287
min,1.000000,100001.000000,2024-07-05 00:00:00,-99514.770000,1.000000,2025-01-01 00:00:00
25%,2255.000000,102255.000000,2024-12-31 00:00:00,-3579.855000,1.000000,2025-03-30 00:00:00
50%,4499.000000,104499.000000,2025-04-01 00:00:00,-1840.575000,2.000000,2025-06-30 00:00:00
75%,6717.000000,106717.000000,2025-07-03 00:00:00,-850.387500,3.000000,2025-10-01 00:00:00
max,9000.000000,109000.000000,2025-12-30 00:00:00,788077.580000,5.000000,2025-12-31 00:00:00
std,2595.050397,2595.050397,NaN,11897.848007,1.221838,NaN


In [1209]:
df_transactions_grouped_on_mrl = df_transactions.copy()
df_transactions_grouped_on_mrl = df_transactions_grouped_on_mrl.groupby('client_id', as_index=False)[['merchant_risk_level']].mean()
df_transactions_grouped_on_mrl.sort_values('merchant_risk_level', ascending=False).tail(2)

,client_id,merchant_risk_level
1136,1707,1.600000
3179,4746,1.592593


In [1210]:
df_transactions_grouped_on_mrl['merchant_risk_level'].mean()

np.float64(2.2838424083085878)

Узнали, что merchant_risk_level присваивается к операции, а не к id. Уже можем использовать сгрупированный критерий merchant_risk_level как потенциальный признак.

In [1211]:
df_transactions["transaction_category"].value_counts(dropna=False)

transaction_category
groceries          52111
transfer           32609
cash_withdrawal    32498
salary             25742
entertainment      22038
loan_payment       21965
utilities          21907
NaN                11503
gambling            8733
Name: count, dtype: int64

In [1212]:
df_transactions_grouped_on_tc_only = df_transactions.copy()
df_transactions_grouped_on_tc_only = df_transactions_grouped_on_tc_only.groupby('transaction_category', as_index=False, dropna=False)[['merchant_risk_level']].mean()
df_transactions_grouped_on_tc_only.sort_values('merchant_risk_level', ascending=False)

,transaction_category,merchant_risk_level
4,loan_payment,2.296998
5,salary,2.295418
1,entertainment,2.294115
7,utilities,2.286872
0,cash_withdrawal,2.281649
6,transfer,2.280486
8,NaN,2.276373
2,gambling,2.274416
3,groceries,2.274266


In [1213]:
df_transactions_grouped_on_tc = df_transactions.copy()
df_transactions_grouped_on_tc = df_transactions_grouped_on_tc.groupby(['client_id','transaction_category'], as_index=False)[['merchant_risk_level']].mean()
df_transactions_grouped_on_tc.sort_values('merchant_risk_level', ascending=False).tail(5)

,client_id,transaction_category,merchant_risk_level
45096,8821,salary,NaN
45184,8838,gambling,NaN
45432,8884,entertainment,NaN
45826,8959,utilities,NaN
45927,8986,transfer,NaN


In [1214]:
gambling_clients = df_transactions.loc[
    df_transactions["transaction_category"].eq("gambling"),
    "client_id"
]

df_transactions_gambl = df_transactions[
    df_transactions["client_id"].isin(gambling_clients)
]

len(df_transactions_gambl), len(df_transactions)

(176230, 229106)

Проверим по флагу гемблинга и отсутсвия флага salary

In [1215]:
df_transactions_gambl = df_transactions_gambl.copy()
df_transactions_gambl = df_transactions_gambl.groupby('transaction_category', as_index=False, dropna=False)[['merchant_risk_level']].mean()
df_transactions_gambl.sort_values('merchant_risk_level', ascending=False)

,transaction_category,merchant_risk_level
4,loan_payment,2.302706
5,salary,2.293075
0,cash_withdrawal,2.288705
7,utilities,2.288483
1,entertainment,2.287825
6,transfer,2.284431
8,NaN,2.275486
2,gambling,2.274416
3,groceries,2.270630


In [1216]:
salary_clients = df_transactions.loc[
    df_transactions["transaction_category"].eq("salary"),
    "client_id"
]

no_salary_clients = df_transactions.loc[
    ~df_transactions["client_id"].isin(salary_clients),
    "client_id"
].drop_duplicates()

df_transactions_no_salary = df_transactions[
    df_transactions["client_id"].isin(no_salary_clients)
]

len(df_transactions_no_salary), len(df_transactions)

(2956, 229106)

In [1217]:
df_transactions_no_salary = df_transactions_no_salary.copy()
df_transactions_no_salary = df_transactions_no_salary.groupby('transaction_category', as_index=False, dropna=False)[['merchant_risk_level']].mean()
df_transactions_no_salary.sort_values('merchant_risk_level', ascending=False)

,transaction_category,merchant_risk_level
1,entertainment,2.269841
5,transfer,2.262712
0,cash_withdrawal,2.245536
3,groceries,2.244635
4,loan_payment,2.238994
6,utilities,2.223602
7,NaN,2.211921
2,gambling,2.132075


In [1218]:
df_transactions[df_transactions['transaction_category'] == 'loan_payment'].head(5)

,client_id,application_id,transaction_date,transaction_category,amount,merchant_risk_level,application_date
9,1,100001,2025-11-23,loan_payment,-955.35,1.0,2025-12-25
15,1,100001,2025-10-19,loan_payment,-3192.53,2.0,2025-12-25
24,1,100001,2025-07-28,loan_payment,-3449.48,1.0,2025-12-25
29,1,100001,2025-11-17,loan_payment,-2223.05,4.0,2025-12-25
31,1,100001,2025-11-15,loan_payment,-1618.64,1.0,2025-12-25


#### 2. 1 Сразу добавим признаки из df_transactions в df, чтобы потом не забыть ничего

In [1219]:
# Вспомогательные столбцы для агрегации
df_transactions["has_gambling"] = (
    df_transactions["transaction_category"]
    .eq("gambling")
    .astype(int)
)

df_transactions["salary_amount"] = (
    df_transactions["amount"]
    .where(
        df_transactions["transaction_category"].eq("salary")
    )
)

df_transactions["loan_payment_amount"] = (
    df_transactions["amount"]
    .where(
        df_transactions["transaction_category"].eq("loan_payment")
    )
)

# Одна строка на клиента
transactions_features = (
    df_transactions
    .groupby("client_id", as_index=False)
    .agg(
        mean_merchant_risk_level=("merchant_risk_level", "mean"),
        has_gambling=("has_gambling", "max"),
        mean_salary=("salary_amount", "mean"),
        mean_loan_payments=("loan_payment_amount", "mean"),
    )
)

# Отношение платежей по займам к зарплате
transactions_features["loan_payments_to_salary"] = (
    transactions_features["mean_loan_payments"].abs()
    / transactions_features["mean_salary"]
)

# Защита от деления на ноль
transactions_features["loan_payments_to_salary"] = (
    transactions_features["loan_payments_to_salary"]
    .replace([np.inf, -np.inf], np.nan)
)
# Добавляем признаки в основную таблицу
df = df.merge(
    transactions_features,
    on="client_id",
    how="left",
    validate="many_to_one",
)

#### 3. Сперва изучаем, затем собираем признаки из bureau

In [1220]:
bureau.head()

,client_id,bureau_account_id,account_type,opened_days_ago,credit_limit,current_balance,max_dpd_last_12m,bureau_status
0,1,B1_0,credit_card,2410.0,20474.70,5760.10,0.0,active
1,2,B2_1,NaN,2983.0,61748.02,47175.10,1.0,active
2,3,B3_2,auto,943.0,93069.06,40725.25,15.0,active
3,3,B3_3,mortgage,2355.0,173852.51,57974.36,7.0,closed
4,3,B3_4,credit_card,302.0,24037.79,7801.91,1.0,closed


account_type — тип кредитного продукта, например: 

    credit_card — кредитная карта; 

    auto — автокредит; 

    mortgage — ипотека.

opened_days_ago — сколько дней назад был открыт этот кредитный счёт относительно некоторой контрольной даты, вероятнее всего даты заявки. Чем больше значение, тем старше счёт.

credit_limit — кредитный лимит или исходный размер кредита. Для карты это обычно лимит, для кредита — вероятно, сумма обязательства.

current_balance — текущая задолженность по счёту. Точный смысл может зависеть от типа продукта.

max_dpd_last_12m — максимальное число дней просрочки за последние 12 месяцев. DPD означает Days Past Due:

bureau_status — текущий статус кредитного счёта, например:

In [1221]:
df_bureau = bureau.drop_duplicates()

In [1222]:
df_bureau['account_type'].value_counts(dropna=False)

account_type
credit_card    8050
consumer       7485
microloan      3619
auto           2546
mortgage       2146
NaN             837
Name: count, dtype: int64

Каждый тип аккаунта несет свой смысл.

credit_card — кредитная карта;

consumer — потребительский кредит;

microloan — микрозайм;

auto — автокредит;

mortgage — ипотека;

NaN — тип кредитного продукта неизвестен или не заполнен.

Кажется, что в этом случае проще достать все столбцы, отдельно выделив тип кредита. Также можно посчитать количество кредитов на аккаунт (с логикой чем больше, тем хуже заемщик). Убираем bureau_account_id. Но придется создать мин, макс, среднее для типов акканутов, потому что один и тот же человек несколь раз может открыть и запрыть ипотеку или авто кредит.

In [1223]:
df_bureau = df_bureau.copy()

# Пропуски в категориях сохраняем как отдельную категорию
df_bureau["account_type"] = df_bureau["account_type"].fillna("unknown")
df_bureau["bureau_status"] = df_bureau["bureau_status"].fillna("unknown")

# 1. Агрегации числовых признаков по клиенту
df_bureau_numeric_features = (
    df_bureau.groupby("client_id", as_index=False)
    .agg(
        df_bureau_total_accounts=("bureau_account_id", "count"),

        df_bureau_opened_days_mean=("opened_days_ago", "mean"),
        df_bureau_opened_days_min=("opened_days_ago", "min"),
        df_bureau_opened_days_max=("opened_days_ago", "max"),

        df_bureau_credit_limit_sum=("credit_limit", "sum"),
        df_bureau_credit_limit_mean=("credit_limit", "mean"),
        df_bureau_credit_limit_max=("credit_limit", "max"),

        df_bureau_current_balance_sum=("current_balance", "sum"),
        df_bureau_current_balance_mean=("current_balance", "mean"),
        df_bureau_current_balance_max=("current_balance", "max"),

        df_bureau_max_dpd_mean=("max_dpd_last_12m", "mean"),
        df_bureau_max_dpd_max=("max_dpd_last_12m", "max"),
    )
)

# 2. Количество кредитов каждого типа
account_type_features = (
    pd.crosstab(df_bureau["client_id"], df_bureau["account_type"])
    .add_prefix("bureau_account_type_")
    .reset_index()
)

# 3. Количество счетов каждого статуса
status_features = (
    pd.crosstab(df_bureau["client_id"], df_bureau["bureau_status"])
    .add_prefix("bureau_status_")
    .reset_index()
)

# 4. Собираем одну таблицу признаков df_bureau
df_bureau_features = (
    df_bureau_numeric_features
    .merge(account_type_features, on="client_id", how="left")
    .merge(status_features, on="client_id", how="left")
)



In [1224]:
df_bureau_features[df_bureau_features['df_bureau_opened_days_mean'] != df_bureau_features['df_bureau_opened_days_min']].head()

,client_id,df_bureau_total_accounts,df_bureau_opened_days_mean,df_bureau_opened_days_min,df_bureau_opened_days_max,df_bureau_credit_limit_sum,df_bureau_credit_limit_mean,df_bureau_credit_limit_max,df_bureau_current_balance_sum,df_bureau_current_balance_mean,df_bureau_current_balance_max,df_bureau_max_dpd_mean,df_bureau_max_dpd_max,bureau_account_type_auto,bureau_account_type_consumer,bureau_account_type_credit_card,bureau_account_type_microloan,bureau_account_type_mortgage,bureau_account_type_unknown,bureau_status_active,bureau_status_closed,bureau_status_unknown
2,3,5,1698.40,302.0,2662.0,413176.18,82635.2360,173852.51,172567.08,34513.416,57974.36,10.600000,30.0,1,1,2,0,1,0,1,3,1
3,4,4,1942.25,972.0,2688.0,333635.71,83408.9275,150745.63,99113.84,24778.460,55673.15,0.500000,1.0,0,3,1,0,0,0,3,1,0
4,5,3,1103.00,184.0,2447.0,88884.53,44442.2650,48948.09,86916.63,28972.210,36383.41,0.333333,1.0,0,2,0,1,0,0,1,1,1
8,9,4,366.25,274.0,545.0,613631.04,153407.7600,201511.86,325114.92,81278.730,118486.65,0.000000,0.0,0,2,2,0,0,0,3,1,0
9,10,5,1144.25,491.0,1812.0,323041.96,64608.3920,174606.77,93048.04,23262.010,46813.20,2.200000,7.0,2,1,1,0,0,1,4,1,0


In [1225]:
# 5. Добавляем признаки в основную таблицу
rows_before = len(df)

df = df.merge(
    df_bureau_features,
    on="client_id",
    how="left",
    validate="many_to_one",
)

assert len(df) == rows_before

#### 4. Сперва изучаем, затем собираем признаки из preveous_loans

In [1226]:
previous_loans.head()

,client_id,previous_loan_id,previous_amount,previous_term_months,closed_days_ago,was_overdue,max_overdue_days
0,2,L2_0,51911.71,3.0,986.0,1.0,3.0
1,2,L2_1,64582.12,24.0,1172.0,0.0,0.0
2,3,L3_2,27102.97,3.0,744.0,0.0,0.0
3,4,L4_3,56399.68,12.0,139.0,1.0,15.0
4,4,L4_4,73117.38,18.0,548.0,0.0,0.0


Кажется, что из таблицы выше можно агрегировать: количество прошлых кредитов на айди, количество просчрочек, отношение количества просрочек к количеству кредитов, максимальный срок просрочки, минимальный closed_days_ago (потому что это сигнал частоты взятия кредитов), минимальный, средний и максимальный previous_term_months, минимальный, средний и максимальный previous_amount.

In [1227]:
df_previous_loans = previous_loans.drop_duplicates().copy()

In [1228]:
previous_loans_features = (
    df_previous_loans
    .groupby("client_id", as_index=False)
    .agg(
        previous_loans_count=("previous_loan_id", "nunique"),

        overdue_loans_count=("was_overdue", "sum"),

        max_overdue_days=("max_overdue_days", "max"),

        min_closed_days_ago=("closed_days_ago", "min"),

        previous_term_months_min=("previous_term_months", "min"),
        previous_term_months_mean=("previous_term_months", "mean"),
        previous_term_months_max=("previous_term_months", "max"),

        previous_amount_min=("previous_amount", "min"),
        previous_amount_mean=("previous_amount", "mean"),
        previous_amount_max=("previous_amount", "max"),
    )
)

previous_loans_features["overdue_loans_ratio"] = (
    previous_loans_features["overdue_loans_count"]
    / previous_loans_features["previous_loans_count"]
)

# На случай деления на ноль
previous_loans_features["overdue_loans_ratio"] = (
    previous_loans_features["overdue_loans_ratio"]
    .replace([np.inf, -np.inf], np.nan)
)

rows_before = len(df)

df = df.merge(
    previous_loans_features,
    on="client_id",
    how="left",
    validate="many_to_one",
)

assert len(df) == rows_before

In [1229]:
previous_loans_features.sort_values('overdue_loans_ratio',ascending=False).head()

,client_id,previous_loans_count,overdue_loans_count,max_overdue_days,min_closed_days_ago,previous_term_months_min,previous_term_months_mean,previous_term_months_max,previous_amount_min,previous_amount_mean,previous_amount_max,overdue_loans_ratio
6865,8977,1,1.0,90.0,161.0,18.0,18.0,18.0,15861.29,15861.29,15861.29,1.0
6864,8976,1,1.0,1.0,982.0,12.0,12.0,12.0,79735.30,79735.30,79735.30,1.0
6861,8972,1,1.0,15.0,240.0,36.0,36.0,36.0,40591.07,40591.07,40591.07,1.0
1700,2188,1,1.0,7.0,1332.0,12.0,12.0,12.0,28515.83,28515.83,28515.83,1.0
1784,2304,1,1.0,15.0,1567.0,24.0,24.0,24.0,73696.74,73696.74,73696.74,1.0


In [1230]:
df["has_bureau_history"] = df["df_bureau_total_accounts"].notna().astype(int)

И туда же добавляем наличие кредитной истории в виде флага.

In [1231]:
df["has_previous_loans"] = df["previous_loans_count"].notna().astype(int)

#### Изучение структуры данных: гипотезы и выводы (1)

1. Таблицы необходимо объединить, чтобы у модели было больше данных для обучения. Но объединение должно быть не прямое, а через получение признаков более высокого уровня в дополнительных таблицых. 
2. Чтобы получить признаки из дополнительных таблиц, необходимо проверить ключи.
3. Обнаружили дупликаты и удалили их train ===> df. 
4. Сделали вывод, что 'client_id' и 'application_id' в df уникальны и не дублируются.
5. Нашли дубли в transactions, поэтому сделали новый чистый файл df_transactions.
6. Выяснили, что для всех client_id есть транзакции. Для 95% есть внешняя кредитная история. И для 77% есть предыдущие займы.
7. Потенциальные признаки из df_transactions: merchant_risk_level по id, флаг операций gambling, avrege salary, avrege loan_payments, соотношение avrege loan_payments к avrege salary. Хоть по merchant_risk подверждений не было, все-равно в дальнейшем добавим эти признаки в общую сводню таблицу.
8. В df_bureau неразбериха со статусом и текущим балансом. Непонятно, что это значит. Использовать осторожно. Кажется, что в этом случае проще достать все столбцы, отдельно выделив тип кредита. Также можно посчитать количество кредитов на аккаунт (с логикой чем больше, тем хуже заемщик). По итогу добавил почти все колонки.
9. Кажется, что из таблицы выше можно агрегировать: количество прошлых кредитов на айди, количество просчрочек, отношение количества просрочек к количеству кредитов, максимальный срок просрочки, минимальный closed_days_ago (потому что это сигнал частоты взятия кредитов), минимальный, средний и максимальный previous_term_months, минимальный, средний и максимальный previous_amount.
10. Итак, все признаки собраны в один главный df. Дальше анализиуем его.
11. Дальше возникла ошибка при попытке построить бэйзлайн на основе решающего дерева, потому что мы не у всех клиенов есть кредитная история или бюро-информация, соответственно, эти признаки я не обработал. Поэтому хоршим сигналом будет наличие бюро-истории в качестве флага в самом df. Добавим ее в 299 ячейке.

### EDA (анализируем основной подготовленный df) | 2 Часть

Проверяем структуру итоговой таблицы. 

In [1232]:
print("Размер:", df.shape)
print("Дубликаты application_id:", df["application_id"].duplicated().sum())
print("Пропуски target:", df["target"].isna().sum())

df.info()

Размер: (6480, 64)
Дубликаты application_id: 0
Пропуски target: 0
<class 'pandas.DataFrame'>
RangeIndex: 6480 entries, 0 to 6479
Data columns (total 64 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   application_id                   6480 non-null   int64  
 1   client_id                        6480 non-null   int64  
 2   employment_type                  6015 non-null   str    
 3   hash_id                          6064 non-null   float64
 4   region_coefficient_extended      6019 non-null   float64
 5   age                              6031 non-null   float64
 6   incoming_amount                  6048 non-null   float64
 7   internal_decision_code           5108 non-null   str    
 8   monthly_income                   6041 non-null   float64
 9   application_date                 6012 non-null   str    
 10  education                        6032 non-null   str    
 11  requested_product          

In [1233]:
missing = (
    df.isna()
    .mean()
    .sort_values(ascending=False)
)
missing.head(30)

days_until_first_overdue       0.286420
previous_term_months_min       0.252778
previous_term_months_max       0.252778
previous_term_months_mean      0.252778
min_closed_days_ago            0.247994
max_overdue_days               0.247685
previous_amount_max            0.241975
previous_amount_min            0.241975
previous_amount_mean           0.241975
post_loan_collection_score     0.236883
overdue_loans_count            0.232407
overdue_loans_ratio            0.232407
previous_loans_count           0.232407
internal_decision_code         0.211728
loan_payments_to_salary        0.113426
mean_loan_payments             0.097840
channel                        0.089043
mean_salary                    0.087809
loan_amount                    0.085802
region                         0.084105
loan_term_months               0.081019
marketing_segment              0.078704
dependents                     0.072222
mean_merchant_risk_level       0.072222
has_gambling                   0.072222


Составим список столбцов на удаление (например, технические признак или потенциальны лик данных).

In [1234]:
drop_columns = ['application_id', 'client_id', 'hash_id', 'post_loan_collection_score', 'days_until_first_overdue', "application_date", 'target']

In [1235]:
df.head()

,application_id,client_id,employment_type,hash_id,region_coefficient_extended,age,incoming_amount,internal_decision_code,monthly_income,application_date,education,requested_product,channel,loan_amount,marketing_segment,post_loan_collection_score,region_coefficient,region,interest_rate,months_at_job,dependents,days_until_first_overdue,loan_term_months,siberia_northern_score,target,mean_merchant_risk_level,has_gambling,mean_salary,mean_loan_payments,loan_payments_to_salary,df_bureau_total_accounts,df_bureau_opened_days_mean,df_bureau_opened_days_min,df_bureau_opened_days_max,df_bureau_credit_limit_sum,df_bureau_credit_limit_mean,df_bureau_credit_limit_max,df_bureau_current_balance_sum,df_bureau_current_balance_mean,df_bureau_current_balance_max,df_bureau_max_dpd_mean,df_bureau_max_dpd_max,bureau_account_type_auto,bureau_account_type_consumer,bureau_account_type_credit_card,bureau_account_type_microloan,bureau_account_type_mortgage,bureau_account_type_unknown,bureau_status_active,bureau_status_closed,bureau_status_unknown,previous_loans_count,overdue_loans_count,max_overdue_days,min_closed_days_ago,previous_term_months_min,previous_term_months_mean,previous_term_months_max,previous_amount_min,previous_amount_mean,previous_amount_max,overdue_loans_ratio,has_bureau_history,has_previous_loans
0,107941,7941,employee,61441390.0,1.0,37.0,30401.28,NaN,30401.28,2025-08-11,bachelor,refinance,partner,63464.49,segment_011,800.87,1.0,east,0.3062,31.0,0.0,61.0,24.0,0.336102,1,2.194444,1.0,7468.340000,-3683.087500,0.493160,3.0,1395.666667,1110.0,1594.0,148917.36,49639.120,97605.53,72210.10,24070.033333,52345.49,5.0,15.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,2.0,0.0,1.0,0.0,0.0,62.0,36.0,36.0,36.0,20623.70,20623.700,20623.70,0.0,1,1
1,101163,1163,employee,67321556.0,0.0,60.0,26024.64,manual_review_bad,26024.64,2025-12-29,school,card,mobile,113513.59,segment_067,721.57,1.0,ural,0.1383,90.0,0.0,82.0,6.0,0.560893,1,2.296296,0.0,13623.675000,-1086.100000,0.079722,5.0,1292.750000,821.0,2184.0,702481.49,140496.298,280053.92,269069.01,53813.802000,129698.64,12.4,60.0,0.0,0.0,4.0,0.0,1.0,0.0,5.0,0.0,0.0,1.0,0.0,0.0,1235.0,24.0,24.0,24.0,20284.16,20284.160,20284.16,0.0,1,1
2,100583,583,contractor,26719968.0,0.0,37.0,21389.83,vip,21389.83,2025-11-09,college,cash,NaN,48993.42,NaN,NaN,1.0,west,0.1729,57.0,1.0,999.0,9.0,NaN,0,2.259259,0.0,NaN,-1328.475000,NaN,1.0,329.000000,329.0,329.0,101951.69,101951.690,101951.69,9362.87,9362.870000,9362.87,7.0,7.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,2.0,1.0,7.0,873.0,6.0,6.0,6.0,50876.72,51096.865,51317.01,0.5,1,1
3,104082,4082,NaN,45184358.0,0.0,NaN,31617.26,approved_auto,NaN,2025-02-23,college,card,web,77765.78,segment_089,170.91,1.0,ural,0.3585,19.0,0.0,999.0,36.0,NaN,0,2.078947,0.0,18155.436667,-752.483333,0.041447,1.0,2320.000000,2320.0,2320.0,84660.99,84660.990,84660.99,59731.91,59731.910000,59731.91,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,NaN,769.0,24.0,24.0,24.0,141115.00,141115.000,141115.00,0.0,1,1
4,108413,8413,employee,58267524.0,0.0,60.0,18000.00,approved_auto,18000.00,2025-12-07,bachelor,card,mobile,97254.10,segment_063,NaN,1.0,west,0.1472,405.0,0.0,730.0,9.0,0.741053,0,2.125000,1.0,3257.700000,-895.112500,0.274768,1.0,2420.000000,2420.0,2420.0,118437.95,118437.950,118437.95,47464.27,47464.270000,47464.27,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1059.0,24.0,24.0,24.0,37376.14,37376.140,37376.14,0.0,1,1


Спорные признаки: 

region_coefficient_extended — вероятно, расширенный региональный коэффициент риска/надбавки, связанный с регионом клиента.

internal_decision_code — вероятно, внутренний код решения скоринговой/андеррайтинговой системы по заявке. 

post_loan_collection_score — скоринговая оценка клиента после выдачи займа, связанная с вероятностью возврата долга или успешного взыскания при просрочке. Collection score обычно используют для оценки вероятности погашения задолженности и выбора стратегии взыскания

days_until_first_overdue — вероятно, количество дней от выдачи займа до первой просрочки.

In [1236]:
test['days_until_first_overdue'].describe()

count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: days_until_first_overdue, dtype: float64

post_loan_collection_score, days_until_first_overdue - лик данных, поскольку в тесте нет никакой информации.

In [1237]:
test.head()

,application_id,client_id,region_coefficient_extended,loan_term_months,siberia_northern_score,hash_id,loan_amount,post_loan_collection_score,months_at_job,channel,education,internal_decision_code,monthly_income,region,employment_type,days_until_first_overdue,incoming_amount,age,dependents,interest_rate,marketing_segment,region_coefficient,requested_product,application_date
0,102531,2531,0.0,18.0,NaN,NaN,73071.51,NaN,38.0,call_center,bachelor,NaN,34183.79,east,employee,NaN,34183.79,NaN,0.0,0.2146,segment_112,1.0,refinance,2025-04-25
1,107213,7213,0.0,12.0,0.317813,30012008.0,65408.73,NaN,23.0,partner,school,NaN,18645.00,north,unemployed,NaN,18645.00,64.0,0.0,0.2851,segment_023,1.0,refinance,2025-05-07
2,100238,238,0.0,36.0,0.902039,92419982.0,NaN,NaN,12.0,office,bachelor,NaN,52931.84,ural,employee,NaN,52931.84,21.0,1.0,0.3810,segment_020,1.0,card,2025-02-26
3,104918,4918,0.0,6.0,0.737848,76137566.0,NaN,NaN,179.0,web,college,NaN,NaN,center,employee,NaN,37301.16,55.0,0.0,0.1471,segment_057,1.0,auto,2025-12-07
4,106480,6480,0.0,12.0,0.746599,21845266.0,76253.20,NaN,153.0,mobile,bachelor,NaN,71928.29,center,NaN,NaN,71928.29,55.0,1.0,0.1840,segment_138,1.0,cash,2025-05-09


Оценим корреляцию числовых признаков с таргетом

In [1238]:
drop_col_after_corr_eval = ['region_coefficient', 'previous_amount_min', 'previous_term_months_max', 'df_bureau_opened_days_mean', 'siberia_northern_score', 'region_coefficient_extended', 'previous_term_months_mean' ]

In [1239]:
X = df.drop(columns=(drop_columns + drop_col_after_corr_eval))
y = df["target"]

In [1240]:
# Корреляционная матрица только для числовых признаков
corr_matrix = df.corr(numeric_only=True)

# Корреляции с target: от большей к меньшей
target_correlations = (
    corr_matrix["target"]
    .drop("target")
    .sort_values(ascending=False)
)


#### Размышления на тему того, как лучше отработать с пропусками

Здесь я проведу дополнительный отсев признаков, опираяься на корреляцию с таргетом, наличие пропусков и заполненность в тесте. А также для каждого таргета подберу лучшее решение для работы с пропусками.

In [1241]:
target_correlations_abs = (
    target_correlations
    .sort_values(key=lambda x: x.abs(), ascending=True)
)

target_correlations_abs

previous_amount_min               -0.000737
previous_term_months_max           0.001920
df_bureau_opened_days_mean        -0.003062
siberia_northern_score            -0.004248
bureau_account_type_unknown       -0.005082
region_coefficient_extended        0.007205
dependents                         0.009075
previous_term_months_mean         -0.011692
loan_payments_to_salary            0.013940
previous_amount_mean               0.014414
bureau_status_unknown              0.017108
hash_id                            0.017678
mean_merchant_risk_level           0.018085
df_bureau_credit_limit_mean       -0.018180
application_id                    -0.020159
client_id                         -0.020159
previous_term_months_min          -0.021136
previous_amount_max                0.023718
df_bureau_credit_limit_max         0.024339
has_previous_loans                 0.025011
bureau_account_type_mortgage       0.029326
df_bureau_current_balance_mean     0.031240
has_gambling                    

In [1242]:
drop_col_after_corr_eval_2 = ['previous_term_months_min', 'min_closed_days_ago', 'previous_amount_mean', 'previous_amount_max', 'internal_decision_code', 'channel', 'df_bureau_credit_limit_mean']

In [1243]:
missing_X = (
    X.isna()
    .mean()
    .sort_values(ascending=False)
)
missing_X.head(30)

previous_term_months_min       0.252778
min_closed_days_ago            0.247994
max_overdue_days               0.247685
previous_amount_mean           0.241975
previous_amount_max            0.241975
overdue_loans_count            0.232407
overdue_loans_ratio            0.232407
previous_loans_count           0.232407
internal_decision_code         0.211728
loan_payments_to_salary        0.113426
mean_loan_payments             0.097840
channel                        0.089043
mean_salary                    0.087809
loan_amount                    0.085802
region                         0.084105
loan_term_months               0.081019
marketing_segment              0.078704
has_gambling                   0.072222
dependents                     0.072222
mean_merchant_risk_level       0.072222
employment_type                0.071759
age                            0.069290
education                      0.069136
monthly_income                 0.067747
incoming_amount                0.066667


In [1244]:
X.head()

,employment_type,age,incoming_amount,internal_decision_code,monthly_income,education,requested_product,channel,loan_amount,marketing_segment,region,interest_rate,months_at_job,dependents,loan_term_months,mean_merchant_risk_level,has_gambling,mean_salary,mean_loan_payments,loan_payments_to_salary,df_bureau_total_accounts,df_bureau_opened_days_min,df_bureau_opened_days_max,df_bureau_credit_limit_sum,df_bureau_credit_limit_mean,df_bureau_credit_limit_max,df_bureau_current_balance_sum,df_bureau_current_balance_mean,df_bureau_current_balance_max,df_bureau_max_dpd_mean,df_bureau_max_dpd_max,bureau_account_type_auto,bureau_account_type_consumer,bureau_account_type_credit_card,bureau_account_type_microloan,bureau_account_type_mortgage,bureau_account_type_unknown,bureau_status_active,bureau_status_closed,bureau_status_unknown,previous_loans_count,overdue_loans_count,max_overdue_days,min_closed_days_ago,previous_term_months_min,previous_amount_mean,previous_amount_max,overdue_loans_ratio,has_bureau_history,has_previous_loans
0,employee,37.0,30401.28,NaN,30401.28,bachelor,refinance,partner,63464.49,segment_011,east,0.3062,31.0,0.0,24.0,2.194444,1.0,7468.340000,-3683.087500,0.493160,3.0,1110.0,1594.0,148917.36,49639.120,97605.53,72210.10,24070.033333,52345.49,5.0,15.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,2.0,0.0,1.0,0.0,0.0,62.0,36.0,20623.700,20623.70,0.0,1,1
1,employee,60.0,26024.64,manual_review_bad,26024.64,school,card,mobile,113513.59,segment_067,ural,0.1383,90.0,0.0,6.0,2.296296,0.0,13623.675000,-1086.100000,0.079722,5.0,821.0,2184.0,702481.49,140496.298,280053.92,269069.01,53813.802000,129698.64,12.4,60.0,0.0,0.0,4.0,0.0,1.0,0.0,5.0,0.0,0.0,1.0,0.0,0.0,1235.0,24.0,20284.160,20284.16,0.0,1,1
2,contractor,37.0,21389.83,vip,21389.83,college,cash,NaN,48993.42,NaN,west,0.1729,57.0,1.0,9.0,2.259259,0.0,NaN,-1328.475000,NaN,1.0,329.0,329.0,101951.69,101951.690,101951.69,9362.87,9362.870000,9362.87,7.0,7.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,2.0,1.0,7.0,873.0,6.0,51096.865,51317.01,0.5,1,1
3,NaN,NaN,31617.26,approved_auto,NaN,college,card,web,77765.78,segment_089,ural,0.3585,19.0,0.0,36.0,2.078947,0.0,18155.436667,-752.483333,0.041447,1.0,2320.0,2320.0,84660.99,84660.990,84660.99,59731.91,59731.910000,59731.91,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,NaN,769.0,24.0,141115.000,141115.00,0.0,1,1
4,employee,60.0,18000.00,approved_auto,18000.00,bachelor,card,mobile,97254.10,segment_063,west,0.1472,405.0,0.0,9.0,2.125000,1.0,3257.700000,-895.112500,0.274768,1.0,2420.0,2420.0,118437.95,118437.950,118437.95,47464.27,47464.270000,47464.27,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1059.0,24.0,37376.140,37376.14,0.0,1,1


max_overdue_days = 0
previous_amount_mean = 0
overdue_loans_count = 0 
previous_loans_count = 0
overdue_loans_ratio = 0
internal_decision_code = 'None'
loan_payments_to_salary = median
mean_loan_payments = median
mean_salary = median
loan_amount = 0
region = None
loan_term_months = 0
marketing_segment = none
dependents = mode
mean_merchant_risk_level = median
has_gambling = 0
employment_type = none
age = median
education = none
monthly_income = median
incoming_amount = median
months_at_job = 0
requested_product = none
df_bureau_credit_limit_max = median
interest_rate = 0





Дальше нужно посмотреть корреляуию между признаками, возмонжо что-то удастся заполнить по взаимокоррлерующим столбцам, а что-то удастся удалить. 

### Попробуем подготовить 1-й бэйзлайн на решающем дереве

Сперва заполним пропуски по схеме выше.

In [1245]:
drop_columns_after_corr = (
    drop_col_after_corr_eval + drop_col_after_corr_eval_2
)

df_reduced = df.drop(
    columns=drop_columns_after_corr,
    errors="ignore",
).copy()

print("Было признаков:", df.shape[1])
print("Стало признаков:", df_reduced.shape[1])

Было признаков: 64
Стало признаков: 50


Также добавляем дополнительно в zero_columns
    "df_bureau_credit_limit_mean",
    "previous_term_months_min",
    "previous_amount_mean",
    "previous_amount_max",


И в median_columns "min_closed_days_ago",

In [1246]:
zero_columns = [
    "max_overdue_days",
    "overdue_loans_count",
    "previous_loans_count",
    "overdue_loans_ratio",
    "loan_amount",
    "loan_term_months",
    "has_gambling",
    "months_at_job",
    "interest_rate",
    
]

median_columns = [
    "loan_payments_to_salary",
    "mean_loan_payments",
    "mean_salary",
    "mean_merchant_risk_level",
    "age",
    "monthly_income",
    "incoming_amount",
    "df_bureau_credit_limit_max",
]

mode_columns = ["dependents"]

unknown_columns = [
    "region",
    "marketing_segment",
    "employment_type",
    "education",
    "requested_product",

]

bureau_columns = [
    "df_bureau_total_accounts",
    "df_bureau_opened_days_min",
    "df_bureau_opened_days_max",
    "df_bureau_credit_limit_sum",
    "df_bureau_current_balance_sum",
    "df_bureau_current_balance_mean",
    "df_bureau_current_balance_max",
    "df_bureau_max_dpd_mean",
    "df_bureau_max_dpd_max",
    "bureau_account_type_auto",
    "bureau_account_type_consumer",
    "bureau_account_type_credit_card",
    "bureau_account_type_microloan",
    "bureau_account_type_mortgage",
    "bureau_account_type_unknown",
    "bureau_status_active",
    "bureau_status_closed",
    "bureau_status_unknown",
]

# Для baseline заполняем отсутствие агрегированной истории нулями
df_reduced[bureau_columns] = df_reduced[bureau_columns].fillna(0)

Импорты

In [1247]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier

Формируем признаки и target

In [1248]:
TARGET = "target"

# Идентификаторы, дата и подозрительные признаки из будущего
drop_columns = [
    TARGET,
    "application_id",
    "client_id",
    "hash_id",
    "application_date",
    "days_until_first_overdue",
    "post_loan_collection_score",
]

X_reduced = df_reduced.drop(columns=drop_columns)
y = df_reduced["target"]

print("Размер X:", X.shape)
print("\nРаспределение target:")
print(y.value_counts(normalize=True))

Размер X: (6480, 50)

Распределение target:
target
0    0.657716
1    0.342284
Name: proportion, dtype: float64


Определяем числовые и категориальные признаки

In [1249]:
categorical_columns = X_reduced.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

numeric_columns = X_reduced.select_dtypes(
    include=["number", "bool"]
).columns.tolist()

print("Числовых признаков:", len(numeric_columns))
print("Категориальных признаков:", len(categorical_columns))

Числовых признаков: 38
Категориальных признаков: 5


Разделяем train и validation

In [1250]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X_reduced,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

print("Train:", X_train.shape)
print("Validation:", X_valid.shape)

Train: (5184, 43)
Validation: (1296, 43)


Создаём preprocessing

In [1251]:
unknown_transformer = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="constant", fill_value="unknown"),
    ),
    (
        "onehot",
        OneHotEncoder(handle_unknown="ignore"),
    ),
])

preprocessor = ColumnTransformer(
    transformers=[
        (
            "zero_fill",
            SimpleImputer(strategy="constant", fill_value=0),
            zero_columns,
        ),
        (
            "median_fill",
            SimpleImputer(strategy="median"),
            median_columns,
        ),
        (
            "mode_fill",
            SimpleImputer(strategy="most_frequent"),
            mode_columns,
        ),
        (
            "unknown_fill",
            unknown_transformer,
            unknown_columns,
        ),
    ],
    remainder="passthrough",
)

Дальше возникили проблемы с препроцессингом, поэтому вернулись в эту часть, чтобы разобраться, в каких колонках пропущена информация и почему.

In [1252]:
nan_columns = X_train.columns[X_train.isna().any()].tolist()

configured_columns = (
    zero_columns
    + median_columns
    + mode_columns
    + unknown_columns
)

unprocessed_nan_columns = [
    col for col in nan_columns
    if col not in configured_columns
]

unprocessed_nan_columns

[]

Все колонки так или иначе связанаы с бюро-информацией, поэтому создадим флан на наличие этой информации для каждого клиента и заполним пропуски нулями.

Создаём и обучаем дерево

In [1253]:
tree_model = DecisionTreeClassifier(
    max_depth=5,
    min_samples_leaf=20,
    random_state=42,
)

tree_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", tree_model),
    ]
)

tree_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](43,)","['employment_type','age','incoming_amount',...,'overdue_loans_ratio', 'has_bureau_history','has_previous_loans']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,43
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('zero_fill', ...), ('median_fill', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of

6. Считаем ROC-AUC

In [1254]:
train_proba = tree_pipeline.predict_proba(X_train)[:, 1]
valid_proba = tree_pipeline.predict_proba(X_valid)[:, 1]

train_roc_auc = roc_auc_score(y_train, train_proba)
valid_roc_auc = roc_auc_score(y_valid, valid_proba)

print(f"Train ROC-AUC: {train_roc_auc:.5f}")
print(f"Valid ROC-AUC: {valid_roc_auc:.5f}")
print(f"Разница:        {train_roc_auc - valid_roc_auc:.5f}")

Train ROC-AUC: 0.80205
Valid ROC-AUC: 0.74755
Разница:        0.05450


Таким образом, результаты решающего дерева:

Train ROC-AUC: 0.80205

Valid ROC-AUC: 0.74755

Разница:       0.05450

### Готовим 1-ый тестовый сабмишен

Строим для test те же признаки

In [1255]:
df_test = test.drop_duplicates().copy()

# Сохраняем исходный порядок заявок
test_application_ids = df_test["application_id"].reset_index(drop=True)

# ============================================================
# TRANSACTIONS
# ============================================================

test_transactions = (
    transactions
    .drop_duplicates()
    .merge(
        df_test[
            [
                "application_id",
                "client_id",
                "application_date",
            ]
        ],
        on=["application_id", "client_id"],
        how="inner",
        validate="many_to_one",
    )
)

test_transactions["transaction_date"] = pd.to_datetime(
    test_transactions["transaction_date"],
    errors="coerce",
)

test_transactions["application_date"] = pd.to_datetime(
    test_transactions["application_date"],
    errors="coerce",
)

# Оставляем только операции до подачи заявки
test_transactions = test_transactions.loc[
    test_transactions["transaction_date"]
    < test_transactions["application_date"]
].copy()

test_transactions["has_gambling"] = (
    test_transactions["transaction_category"]
    .eq("gambling")
    .astype(int)
)

test_transactions["salary_amount"] = (
    test_transactions["amount"]
    .where(test_transactions["transaction_category"].eq("salary"))
)

test_transactions["loan_payment_amount"] = (
    test_transactions["amount"]
    .where(test_transactions["transaction_category"].eq("loan_payment"))
)

test_transactions_features = (
    test_transactions
    .groupby(
        ["application_id", "client_id"],
        as_index=False,
    )
    .agg(
        mean_merchant_risk_level=("merchant_risk_level", "mean"),
        has_gambling=("has_gambling", "max"),
        mean_salary=("salary_amount", "mean"),
        mean_loan_payments=("loan_payment_amount", "mean"),
    )
)

test_transactions_features["loan_payments_to_salary"] = (
    test_transactions_features["mean_loan_payments"].abs()
    / test_transactions_features["mean_salary"]
)

test_transactions_features["loan_payments_to_salary"] = (
    test_transactions_features["loan_payments_to_salary"]
    .replace([np.inf, -np.inf], np.nan)
)

df_test = df_test.merge(
    test_transactions_features,
    on=["application_id", "client_id"],
    how="left",
    validate="one_to_one",
)

# ============================================================
# BUREAU
# ============================================================

df_test = df_test.merge(
    df_bureau_features,
    on="client_id",
    how="left",
    validate="many_to_one",
)

df_test["has_bureau_history"] = (
    df_test["df_bureau_total_accounts"]
    .notna()
    .astype(int)
)

# ============================================================
# PREVIOUS LOANS
# ============================================================

df_test = df_test.merge(
    previous_loans_features,
    on="client_id",
    how="left",
    validate="many_to_one",
)

df_test["has_previous_loans"] = df_test["previous_loans_count"].notna().astype(int)

assert len(df_test) == len(test)
assert df_test["application_id"].duplicated().sum() == 0

print("Размер подготовленного test:", df_test.shape)

Размер подготовленного test: (2520, 63)


Применяем то же удаление признаков

In [1256]:
df_test_reduced = df_test.drop(
    columns=drop_col_after_corr_eval,
    errors="ignore",
).copy()

# Та же обработка bureau-признаков, что использовалась в train
df_test_reduced[bureau_columns] = (
    df_test_reduced[bureau_columns]
    .fillna(0)
)

Формируем матрицу X_test

In [1257]:
test_drop_columns = [
    "application_id",
    "client_id",
    "hash_id",
    "application_date",
    "days_until_first_overdue",
    "post_loan_collection_score",
]

X_test = df_test_reduced.drop(
    columns=test_drop_columns,
    errors="ignore",
)

missing_in_test = sorted(
    set(X_reduced.columns) - set(X_test.columns)
)

extra_in_test = sorted(
    set(X_test.columns) - set(X_reduced.columns)
)

print("Нет в test:", missing_in_test)
print("Лишние в test:", extra_in_test)

assert not missing_in_test, (
    f"В test отсутствуют признаки: {missing_in_test}"
)

# Строго тот же состав и порядок, что у train
X_test = X_test[X_reduced.columns]

print("Train:", X_reduced.shape)
print("Test: ", X_test.shape)

Нет в test: []
Лишние в test: ['channel', 'df_bureau_credit_limit_mean', 'internal_decision_code', 'min_closed_days_ago', 'previous_amount_max', 'previous_amount_mean', 'previous_term_months_min']
Train: (6480, 43)
Test:  (2520, 43)


Переобучаем модель на всём train

In [1258]:
from sklearn.base import clone

final_tree_pipeline = clone(tree_pipeline)

final_tree_pipeline.fit(
    X_reduced,
    y,
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](43,)","['employment_type','age','incoming_amount',...,'overdue_loans_ratio', 'has_bureau_history','has_previous_loans']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,43
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('zero_fill', ...), ('median_fill', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of

Получаем вероятности

Для ROC-AUC нужны именно вероятности класса 1, а не ответы 0/1:

In [1259]:
test_proba = final_tree_pipeline.predict_proba(X_test)[:, 1]

print("Количество прогнозов:", len(test_proba))


Количество прогнозов: 2520


Создаём submisson.csv

In [1260]:
submission = pd.DataFrame(
    {
        "application_id": test_application_ids,
        "target": test_proba,
    }
)

assert submission.shape == (len(test), 2)
assert submission["application_id"].isna().sum() == 0
assert submission["application_id"].duplicated().sum() == 0
assert submission["target"].isna().sum() == 0

SUBMISSIONS_DIR = ROOT / "submissions"
SUBMISSIONS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

submission_path = SUBMISSIONS_DIR / "submission.csv"

submission.to_csv(
    submission_path,
    index=False,
)

print("Файл сохранён:", submission_path)
submission.head(10)

Файл сохранён: c:\temp\shift_credit_scoring\submissions\submission.csv


,application_id,target
0,102531,0.727273
1,107213,0.490099
2,100238,0.435484
3,104918,0.107027
4,106480,0.228571
5,101134,0.148148
6,105902,0.107027
7,108091,0.259146
8,104260,0.107027
9,108432,0.107027


Результат на тесте 0.67

### Тюнинг бэйзлайна-решающего дерева (подберем гипрепараметры и порог классификации)

Быстро переберем минимально количество объектов в листе и глубину дерева. 

In [1261]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold



In [1262]:
tree_grid_pipeline  = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            DecisionTreeClassifier(
                random_state=42,
            ),
        ),
    ]
)

Задаём пространство параметров

In [1263]:
tree_param_grid = {
    "model__max_depth": [
        2,
        3,
        4,
        5,
        6,
        7,
        8,
        10,
        12,
        15,
        None,
    ],

    "model__min_samples_leaf": [
        1,
        2,
        5,
        10,
        15,
        20,
        30,
        40,
        60,
        80,
        100,
    ],
}

Стратифицированная кросс-валидация

In [1264]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

Полный перебор

In [1265]:
tree_grid_search = GridSearchCV(
    estimator=tree_grid_pipeline,
    param_grid=tree_param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    return_train_score=True,
    refit=True,
)

tree_grid_search.fit(
    X_train,
    y_train,
)

Fitting 5 folds for each of 121 candidates, totalling 605 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__max_depth': [2, 3, ...], 'model__min_samples_leaf': [1, 2, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"return_train_score return_train_score: bool, default=FalseIf ``False``, the ``cv_results_`` attribute will not include trainingscores.Computing training scores is used to get insights on how differentparameter settings impact the overfitting/underfitting trade-off.However computing the scores on the training set can be computationallyexpensive and is not strictly required to select the parameters thatyield the best generalization performance... versionadded:: 0.19.. versionchanged:: 0.21 Default value was changed from ``True`` to ``False``",True
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the re

Результаты подбора

In [1266]:
print("Лучший средний ROC-AUC на CV:")
print(tree_grid_search.best_score_)

print("\nЛучшие параметры:")
print(tree_grid_search.best_params_)

Лучший средний ROC-AUC на CV:
0.7769150316182097

Лучшие параметры:
{'model__max_depth': 7, 'model__min_samples_leaf': 100}


In [1282]:
best_index = tree_grid_search.best_index_

best_cv_train_auc = tree_grid_search.cv_results_[
    "mean_train_score"
][best_index]

best_cv_valid_auc = tree_grid_search.cv_results_[
    "mean_test_score"
][best_index]

print(best_index), print(best_cv_train_auc), print(best_cv_valid_auc)

65
0.814830799714828
0.7769150316182097


(None, None, None)

Лучший средний ROC-AUC на CV у тюнингованного решающего дерева:
0.7769150316182097

### 2-й Бэйзлайн на логистической регрессии

Создаём pipeline

In [1268]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

In [1269]:
logreg_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor,
        ),
        (
            "scaler",
            StandardScaler(
                with_mean=False,
            ),
        ),
        (
            "model",
            LogisticRegression(
                C=1.0,
                penalty="l2",
                solver="liblinear",
                max_iter=2000,
                random_state=42,
            ),
        ),
    ]
)

Обучаем baseline

In [1270]:
logreg_pipeline.fit(
    X_train,
    y_train,
)

c:\temp\shift_credit_scoring\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('scaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](43,)","['employment_type','age','incoming_amount',...,'overdue_loans_ratio', 'has_bureau_history','has_previous_loans']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,43
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('zero_fill', ...), ('median_fill', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (defa

Считаем ROC-AUC

In [1271]:
train_proba_logreg = logreg_pipeline.predict_proba(
    X_train
)[:, 1]

valid_proba_logreg = logreg_pipeline.predict_proba(
    X_valid
)[:, 1]

In [1272]:
train_auc_logreg = roc_auc_score(
    y_train,
    train_proba_logreg,
)

valid_auc_logreg = roc_auc_score(
    y_valid,
    valid_proba_logreg,
)

print("Logistic Regression")
print("Train ROC-AUC:", train_auc_logreg)
print("Valid ROC-AUC:", valid_auc_logreg)

Logistic Regression
Train ROC-AUC: 0.8457430066751083
Valid ROC-AUC: 0.7881508268832212


Логистическая регрессия показала хорошие результаты, превзойдя тюнингованно дерево.

Train ROC-AUC: 0.9525690405895518
Valid ROC-AUC: 0.9245389755953136

Сравниваем с прошлымы моделями

In [1273]:
comparison = pd.DataFrame(
    {
        "model": [
            "Decision Tree",
            "Decision Tree Tuned",
            "Logistic Regression",
        ],
        "train_roc_auc": [
            train_roc_auc,
            best_cv_train_auc,
            train_auc_logreg,
        ],
        "valid_roc_auc": [
            valid_roc_auc,
            best_cv_valid_auc,
            valid_auc_logreg,
        ],
    }
)

comparison

,model,train_roc_auc,valid_roc_auc
0,Decision Tree,0.802048,0.747551
1,Decision Tree Tuned,0.814831,0.776915
2,Logistic Regression,0.845743,0.788151


### Тюнинг логистической регрессии

Тюнингуем три наиболее значимых параметра:

C — сила регуляризации;

penalty — L1 или L2;

class_weight — обычные или сбалансированные веса классов.

In [1274]:
logreg_grid_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor,
        ),
        (
            "scaler",
            StandardScaler(
                with_mean=False,
            ),
        ),
        (
            "model",
            LogisticRegression(
                solver="liblinear",
                max_iter=5000,
                random_state=42,
            ),
        ),
    ]
)

Сетка параметров

In [1275]:
logreg_param_grid = {
    "model__C": [
        0.001,
        0.003,
        0.01,
        0.03,
        0.1,
        0.3,
        1.0,
        3.0,
        10.0,
        30.0,
        100.0,
        300.0,
        1000.0,
    ],

    "model__penalty": [
        "l1",
        "l2",
    ],

    "model__class_weight": [
        None,
        "balanced",
    ],
}

Полный поиск

In [1276]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

logreg_grid_search = GridSearchCV(
    estimator=logreg_grid_pipeline,
    param_grid=logreg_param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    return_train_score=True,
    refit=True,
)

logreg_grid_search.fit(
    X_train,
    y_train,
)

Fitting 5 folds for each of 52 candidates, totalling 260 fits


c:\temp\shift_credit_scoring\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\temp\shift_credit_scoring\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...liblinear'))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__C': [0.001, 0.003, ...], 'model__class_weight': [None, 'balanced'], 'model__penalty': ['l1', 'l2']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"return_train_score return_train_score: bool, default=FalseIf ``False``, the ``cv_results_`` attribute will not include trainingscores.Computing training scores is used to get insights on how differentparameter settings impact the overfitting/underfitting trade-off.However computing the scores on the training set can be computationallyexpensive and is not strictly required to select the parameters thatyield the best generalization performance... versionadded:: 0.19.. versionchanged:: 0.21 Default value was changed from ``True`` to ``False``",True
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_param

Лучшие параметры

In [1277]:
print("Лучший средний ROC-AUC на CV:")
print(logreg_grid_search.best_score_)

print("\nЛучшие параметры:")
print(logreg_grid_search.best_params_)

Лучший средний ROC-AUC на CV:
0.820999655337129

Лучшие параметры:
{'model__C': 0.03, 'model__class_weight': 'balanced', 'model__penalty': 'l1'}


Train и holdout validation

In [1278]:
best_logreg_pipeline = logreg_grid_search.best_estimator_

train_proba_logreg_tuned = (
    best_logreg_pipeline.predict_proba(X_train)[:, 1]
)

valid_proba_logreg_tuned = (
    best_logreg_pipeline.predict_proba(X_valid)[:, 1]
)

train_auc_logreg_tuned = roc_auc_score(
    y_train,
    train_proba_logreg_tuned,
)

valid_auc_logreg_tuned = roc_auc_score(
    y_valid,
    valid_proba_logreg_tuned,
)

print("Tuned Logistic Regression")
print("Train ROC-AUC:", train_auc_logreg_tuned)
print("Valid ROC-AUC:", valid_auc_logreg_tuned)

Tuned Logistic Regression
Train ROC-AUC: 0.8363223095412062
Valid ROC-AUC: 0.8021322378716745


CV-оценки лучшей комбинации

In [1279]:
best_index = logreg_grid_search.best_index_

best_cv_train_auc = (
    logreg_grid_search.cv_results_[
        "mean_train_score"
    ][best_index]
)

best_cv_valid_auc = (
    logreg_grid_search.cv_results_[
        "mean_test_score"
    ][best_index]
)

best_cv_std = (
    logreg_grid_search.cv_results_[
        "std_test_score"
    ][best_index]
)

print("Средний CV Train ROC-AUC:", best_cv_train_auc)
print("Средний CV Validation ROC-AUC:", best_cv_valid_auc)
print("Разброс Validation ROC-AUC:", best_cv_std)

Средний CV Train ROC-AUC: 0.8367709882670876
Средний CV Validation ROC-AUC: 0.820999655337129
Разброс Validation ROC-AUC: 0.016467785024516218


Добавляем модель в сравнение

In [1283]:
comparison = pd.DataFrame(
    {
        "model": [
            "Decision Tree",
            "Decision Tree Tuned",
            "Logistic Regression",
            "Logistic Regression Tuned",
        ],
        "train_roc_auc": [
            0.80205,
            0.814830799714828,
            0.8457430066751083,
            train_auc_logreg_tuned,
        ],
        "valid_roc_auc": [
            0.74755,
            0.7769150316182097,
            0.7881508268832212,
            valid_auc_logreg_tuned,
        ],
    }
)

comparison["auc_gap"] = (
    comparison["train_roc_auc"]
    - comparison["valid_roc_auc"]
)

comparison

,model,train_roc_auc,valid_roc_auc,auc_gap
0,Decision Tree,0.802050,0.747550,0.054500
1,Decision Tree Tuned,0.814831,0.776915,0.037916
2,Logistic Regression,0.845743,0.788151,0.057592
3,Logistic Regression Tuned,0.836322,0.802132,0.034190
